# Portfolio Tracker — Notebook Interactivo

Réplica completa de la aplicación web usando `PortfolioClient` (todos los métodos devuelven DataFrames).

**Secciones:**
1. Setup
2. Carga de datos
3. Resumen (KPIs)
4. Posiciones + Lotes abiertos
5. Movimientos + Resumen de Órdenes
6. Asset Allocation
7. **Evolución Real del Patrimonio** (basada en órdenes)
8. **Evolución Real por Fondo**
9. Benchmark vs MSCI World
10. Detalle Individual de Fondo
11. Evolución Temporal (NAV base 100)
12. Métricas de Evolución por Fondo
13. Retornos Anuales (Heatmap)
14. Correlaciones
15. Simulador: Añadir Fondo
16. Simulador: Rebalanceo
17. Proyección What-If
18. Tax Optimizer (FIFO)
19. Análisis de Traspasos
20. Performance + Diagnósticos

## 1. Setup

In [1]:

# ── Magics ────────────────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2
%matplotlib inline

# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

from IPython.display import display, HTML   # display() nativo — renderiza DataFrames enriquecidos

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

# ── Path ──────────────────────────────────────────────────────────────────────
# Usa la ubicación real del notebook, no el cwd del proceso
BACKEND_DIR = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from app.client import PortfolioClient  # noqa: E402  (import late a propósito)

print(f"✅ Backend dir : {BACKEND_DIR}")
print(f"✅ Python path : {sys.path[0]}")
print(f"✅ PortfolioClient importado")


✅ Backend dir : c:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend
✅ Python path : c:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend
✅ PortfolioClient importado


## 2. Carga de Datos

Cargamos el fichero de órdenes TSV y creamos el `PortfolioClient` que centraliza
todos los accesos a datos (precios, sectores, regiones, métricas, etc.).

In [2]:
TSV_PATH = str(BACKEND_DIR / "data" / "Órdenes 1238478.tsv")
client = PortfolioClient(TSV_PATH)
print(client)

# Vista rápida de los movimientos cargados
mov = client.movements()
print(f"\nMovimientos cargados: {len(mov)}")
mov.tail(10)

PortfolioClient(18 positions)

Movimientos cargados: 65


,Fecha,ISIN,Importe,Participaciones,Estado
55,2026-02-24,LU3038481936,"2,000.00",6.94,Finalizada
56,2026-02-24,LU0329355670,500.00,1.38,Finalizada
57,2026-02-24,IE00B88WFS66,500.00,56.13,Finalizada
58,2026-02-25,LU0273159177,"2,019.02",5.38,Finalizada
59,2026-02-25,IE00BH6XSF26,"1,009.51",3.17,Finalizada
60,2026-03-24,LU1598719752,500.00,2.98,Finalizada
61,2026-03-24,IE00BH6XSF26,500.00,1.66,Finalizada
62,2026-03-24,IE000ZYRH0Q7,"1,000.00",96.14,Finalizada
63,2026-04-15,LU3038481936,"1,500.00",5.32,Finalizada
64,2026-04-26,IE000ZYRH0Q7,"1,000.00",89.46,Finalizada


## 2b. Fuentes extra de órdenes: MyInvestor ETF + Trade Republic

Cargamos y visualizamos los ficheros adicionales de ETCs/ETFs antes de inyectarlos en el portfolio.


In [3]:
import importlib
import app.services.core_portfolio as _cp_mod
importlib.reload(_cp_mod)

from app.services.core_portfolio import Portfolio
from app.services.portfolio_service import (
    MYINVESTOR_ETF_PATH,
    TRADEREPUBLIC_CSV_PATH,
)

# ── Trade Republic ────────────────────────────────────────────────────────────
print("=" * 65)
print("TRADE REPUBLIC — Exportación de transacción.csv")
print("=" * 65)
if TRADEREPUBLIC_CSV_PATH.exists():
    df_tr = Portfolio._normalize_traderepublic_df(str(TRADEREPUBLIC_CSV_PATH))
    print(f"  Filas cargadas (TRADING):  {len(df_tr)}")
    print(f"  ISINs distintos:           {df_tr['ISIN'].nunique()}")
    print(f"  Rango de fechas:           {df_tr['Fecha'].min().date()} → {df_tr['Fecha'].max().date()}")
    print(f"  Importe total invertido:   €{df_tr['Importe'].sum():,.2f}")
    print()
    print("  ISINs y nombres:")
    print(df_tr.groupby("ISIN")["Fondo"].first().reset_index().to_string(index=False))
    print()
    display(df_tr.head(10))
else:
    print(f"  ⚠️  Fichero no encontrado: {TRADEREPUBLIC_CSV_PATH}")

# ── MyInvestor ETF ────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("MYINVESTOR ETF — MyInvestorETF.xlsx")
print("=" * 65)
if MYINVESTOR_ETF_PATH.exists():
    df_mi = Portfolio._normalize_myinvestor_etf_df(str(MYINVESTOR_ETF_PATH))
    print(f"  Filas cargadas:            {len(df_mi)}")
    print(f"  ISINs distintos:           {df_mi['ISIN'].nunique()}")
    print(f"  Rango de fechas:           {df_mi['Fecha'].min().date()} → {df_mi['Fecha'].max().date()}")
    print(f"  Importe total invertido:   €{df_mi['Importe'].sum():,.2f}")
    print()
    print("  ISINs y nombres:")
    print(df_mi.groupby("ISIN")["Fondo"].first().reset_index().to_string(index=False))
    print()
    display(df_mi.head(10))
else:
    print(f"  ⚠️  Fichero no encontrado: {MYINVESTOR_ETF_PATH}")


TRADE REPUBLIC — Exportación de transacción.csv
  Filas cargadas (TRADING):  19
  ISINs distintos:           1
  Rango de fechas:           2025-09-24 → 2026-05-04
  Importe total invertido:   €6,013.15

  ISINs y nombres:
        ISIN                   Fondo
IE00B4ND3602 Physical Gold USD (Acc)



,ISIN,Fecha,Participaciones,Importe,Estado,Tipo,Fondo
16,IE00B4ND3602,2025-09-24,32.00,"1,972.96",Finalizada,Compra,Physical Gold USD (Acc)
17,IE00B4ND3602,2025-09-24,0.44,26.88,Finalizada,Compra,Physical Gold USD (Acc)
19,IE00B4ND3602,2025-10-02,15.62,"1,000.00",Finalizada,Compra,Physical Gold USD (Acc)
22,IE00B4ND3602,2025-10-16,7.10,500.00,Finalizada,Compra,Physical Gold USD (Acc)
24,IE00B4ND3602,2025-11-03,3.70,250.00,Finalizada,Compra,Physical Gold USD (Acc)
25,IE00B4ND3602,2025-11-17,3.67,250.00,Finalizada,Compra,Physical Gold USD (Acc)
27,IE00B4ND3602,2025-12-02,2.85,200.00,Finalizada,Compra,Physical Gold USD (Acc)
28,IE00B4ND3602,2025-12-16,2.83,200.00,Finalizada,Compra,Physical Gold USD (Acc)
30,IE00B4ND3602,2026-01-02,2.75,200.00,Finalizada,Compra,Physical Gold USD (Acc)
31,IE00B4ND3602,2026-01-16,2.60,200.00,Finalizada,Compra,Physical Gold USD (Acc)



MYINVESTOR ETF — MyInvestorETF.xlsx
  Filas cargadas:            25
  ISINs distintos:           1
  Rango de fechas:           2025-01-22 → 2026-02-06
  Importe total invertido:   €8,426.08

  ISINs y nombres:
        ISIN        Fondo
XS2940466316 XS2940466316



,ISIN,Fecha,Participaciones,Importe,Estado,Tipo,Fondo
0,XS2940466316,2025-10-07,75.00,744.37,Finalizada,Compra,XS2940466316
1,XS2940466316,2025-10-09,50.00,524.78,Finalizada,Compra,XS2940466316
2,XS2940466316,2025-10-13,80.00,760.91,Finalizada,Compra,XS2940466316
3,XS2940466316,2025-10-14,30.00,289.25,Finalizada,Compra,XS2940466316
4,XS2940466316,2025-10-17,60.00,547.71,Finalizada,Compra,XS2940466316
5,XS2940466316,2025-10-17,35.00,317.66,Finalizada,Compra,XS2940466316
6,XS2940466316,2025-10-17,50.00,445.53,Finalizada,Compra,XS2940466316
7,XS2940466316,2025-10-17,25.00,225.27,Finalizada,Compra,XS2940466316
8,XS2940466316,2025-11-05,30.00,266.24,Finalizada,Compra,XS2940466316
9,XS2940466316,2025-11-06,25.00,220.26,Finalizada,Compra,XS2940466316


In [4]:
# Resumen consolidado de todas las fuentes de órdenes
print("=" * 65)
print("RESUMEN CONSOLIDADO DE TODAS LAS FUENTES")
print("=" * 65)

# Movimientos del TSV principal (ya en el kernel como 'mov')
df_tsv = mov.copy()
df_tsv["Fuente"] = "TSV (Fondos Indexa)"

frames = [df_tsv]
if "df_tr" in dir() and not df_tr.empty:
    _dtr = df_tr.copy(); _dtr["Fuente"] = "Trade Republic (ETC)"
    frames.append(_dtr)
if "df_mi" in dir() and not df_mi.empty:
    _dmi = df_mi.copy(); _dmi["Fuente"] = "MyInvestor (ETF)"
    frames.append(_dmi)

mov_all = pd.concat(frames, ignore_index=True)
mov_all["Fecha"] = pd.to_datetime(mov_all["Fecha"])

print(f"\nTotal órdenes:           {len(mov_all)}")
print(f"ISINs únicos totales:    {mov_all['ISIN'].nunique()}")
print(f"Importe total:           €{mov_all['Importe'].sum():,.2f}")
print()

# Desglose por fuente
summary_fuente = mov_all.groupby("Fuente").agg(
    Órdenes=("ISIN", "count"),
    ISINs=("ISIN", "nunique"),
    Importe_Total=("Importe", "sum"),
    Desde=("Fecha", "min"),
    Hasta=("Fecha", "max"),
)
summary_fuente["Desde"] = summary_fuente["Desde"].dt.strftime("%Y-%m-%d")
summary_fuente["Hasta"] = summary_fuente["Hasta"].dt.strftime("%Y-%m-%d")
display(summary_fuente.style.format({"Importe_Total": "€{:,.2f}"}))

print()
# Tabla por ISIN (todas las fuentes)
fondo_col = "Fondo" if "Fondo" in mov_all.columns else "ISIN"
agg_dict = {"Fuente": ("Fuente", "first"), "Órdenes": ("ISIN", "count"),
            "Participaciones": ("Participaciones", "sum"), "Importe": ("Importe", "sum")}
if fondo_col == "Fondo":
    agg_dict["Nombre"] = ("Fondo", "first")

resumen_isin = (
    mov_all.groupby("ISIN")
    .agg(**agg_dict)
    .sort_values("Importe", ascending=False)
)
display(resumen_isin.style.format({"Participaciones": "{:,.4f}", "Importe": "€{:,.2f}"}))


RESUMEN CONSOLIDADO DE TODAS LAS FUENTES

Total órdenes:           109
ISINs únicos totales:    21
Importe total:           €147,523.60



,Órdenes,ISINs,Importe_Total,Desde,Hasta
Fuente,,,,,
MyInvestor (ETF),25,1,"€8,426.08",2025-01-22,2026-02-06
TSV (Fondos Indexa),65,19,"€133,084.37",2024-10-27,2026-04-26
Trade Republic (ETC),19,1,"€6,013.15",2025-09-24,2026-05-04


,Fuente,Órdenes,Participaciones,Importe,Nombre
ISIN,,,,,
IE00BYX5MX67,TSV (Fondos Indexa),22,"1,849.2990","€30,956.08",nan
IE00BD0NCM55,TSV (Fondos Indexa),1,689.9000,"€16,402.45",nan
IE00BYX5NX33,TSV (Fondos Indexa),1,"1,332.1350","€16,141.22",nan
LU0302296495,TSV (Fondos Indexa),9,"3,122.0180","€11,000.00",nan
XS2940466316,MyInvestor (ETF),25,"1,050.0000","€8,426.08",XS2940466316
LU0273159177,TSV (Fondos Indexa),4,20.9250,"€7,019.02",nan
IE00BH6XSF26,TSV (Fondos Indexa),3,21.4110,"€6,509.51",nan
IE00B4ND3602,Trade Republic (ETC),19,89.0289,"€6,013.15",Physical Gold USD (Acc)
LU1598719752,TSV (Fondos Indexa),2,32.8830,"€5,500.00",nan


## 3. Resumen (KPIs)

In [5]:
pos = client.positions(live=True)

total_valor = pos["Valor_Actual"].sum()
total_inv = pos["Capital_Invertido"].sum()
ganancia = total_valor - total_inv
ganancia_pct = (ganancia / total_inv * 100) if total_inv > 0 else 0

print("═" * 60)
print(f"{'RESUMEN DE CARTERA':^60}")
print("═" * 60)
print(f"  Patrimonio:        €{total_valor:>12,.2f}")
print(f"  Capital Invertido: €{total_inv:>12,.2f}")
print(f"  Ganancia:          €{ganancia:>12,.2f} ({ganancia_pct:+.2f}%)")
print(f"  Nº Fondos:         {len(pos)}")
print("═" * 60)

════════════════════════════════════════════════════════════
                     RESUMEN DE CARTERA                     
════════════════════════════════════════════════════════════
  Patrimonio:        €  145,995.25
  Capital Invertido: €  132,895.46
  Ganancia:          €   13,099.79 (+9.86%)
  Nº Fondos:         18
════════════════════════════════════════════════════════════


In [6]:
pos

,ISIN,Fondo,Valor_Actual,Capital_Invertido,Ganancia_Euros,Ganancia_Pct,Participaciones,Precio_Actual,Fecha_NAV
0,IE00BYX5MX67,Fidelity S&P 500 Index Fund,"28,398.21","25,002.88","3,395.33",13.58,"1,849.30",15.36,2026-05-07
1,IE00BD0NCM55,iShares Developed World Index Fund (IE),"18,465.86","16,402.45","2,063.41",12.58,689.90,26.77,2026-05-07
2,IE00BYX5NX33,Fidelity MSCI World Index Fund,"17,708.47","16,141.22","1,567.25",9.71,"1,332.13",13.29,2026-05-07
3,LU0302296495,DNB Fund - Technology,"12,551.74","11,000.00","1,551.74",14.11,7.14,"1,758.93",2026-05-07
4,ES0146309002,Horos Value Internacional FI,"11,887.31","10,820.38","1,066.94",9.86,54.52,218.02,2026-05-07
5,LU0273159177,DWS Invest - Gold and Precious Metals Equities,"6,784.72","7,019.02",-234.30,-3.34,20.93,324.24,2026-05-07
6,IE00BH6XSF26,Heptagon Fund ICAV - Kopernik Global All-Cap E...,"6,800.30","6,509.51",290.79,4.47,21.41,317.61,2026-05-07
7,LU1598719752,Cobas LUX SICAV - Cobas International Fund,"5,865.01","5,500.00",365.01,6.64,32.88,178.36,2026-05-07
8,LU2466448532,Echiquier Space,"5,618.48","5,000.00",618.48,12.37,28.18,199.35,2026-05-07
9,LU0840158819,Storm Fund II - Storm Bond Fund,"5,076.70","5,000.00",76.70,1.53,32.27,157.30,2026-05-08


## 4. Posiciones + Lotes Abiertos

In [ ]:
pos

,ISIN,Fondo,Valor_Actual,Capital_Invertido,Ganancia_Euros,Ganancia_Pct,Participaciones,Precio_Actual,Fecha_NAV
0,IE00BYX5MX67,Fidelity S&P 500 Index Fund,"28,398.21","25,002.88","3,395.33",13.58,"1,849.30",15.36,2026-05-07
1,IE00BD0NCM55,iShares Developed World Index Fund (IE),"18,465.86","16,402.45","2,063.41",12.58,689.90,26.77,2026-05-07
2,IE00BYX5NX33,Fidelity MSCI World Index Fund,"17,708.47","16,141.22","1,567.25",9.71,"1,332.13",13.29,2026-05-07
3,LU0302296495,DNB Fund - Technology,"12,551.74","11,000.00","1,551.74",14.11,7.14,"1,758.93",2026-05-07
4,ES0146309002,Horos Value Internacional FI,"11,887.31","10,820.38","1,066.94",9.86,54.52,218.02,2026-05-07
5,LU0273159177,DWS Invest - Gold and Precious Metals Equities,"6,784.72","7,019.02",-234.30,-3.34,20.93,324.24,2026-05-07
6,IE00BH6XSF26,Heptagon Fund ICAV - Kopernik Global All-Cap E...,"6,800.30","6,509.51",290.79,4.47,21.41,317.61,2026-05-07
7,LU1598719752,Cobas LUX SICAV - Cobas International Fund,"5,865.01","5,500.00",365.01,6.64,32.88,178.36,2026-05-07
8,LU2466448532,Echiquier Space,"5,618.48","5,000.00",618.48,12.37,28.18,199.35,2026-05-07
9,LU0840158819,Storm Fund II - Storm Bond Fund,"5,076.70","5,000.00",76.70,1.53,32.27,157.30,2026-05-08


In [ ]:
# Lotes abiertos (FIFO)
lots = client.open_lots()
print(f"Lotes abiertos: {len(lots)}")
lots.head(20)

Lotes abiertos: 59


,ISIN,Fondo,Fecha_Compra,Participaciones_Iniciales,Participaciones_Restantes,Importe_Invertido,Precio_Compra_Unitario
0,ES0140794001,ES0140794001,2025-08-27,152.59,152.59,"2,000.00",13.11
1,ES0141116030,ES0141116030,2025-10-04,7.97,7.97,"2,000.00",250.97
2,ES0146309002,ES0146309002,2025-09-21,54.52,54.52,"10,820.38",198.45
3,IE000ZYRH0Q7,IE000ZYRH0Q7,2025-12-31,92.62,92.62,"1,000.00",10.80
4,IE000ZYRH0Q7,IE000ZYRH0Q7,2026-03-24,96.14,96.14,"1,000.00",10.40
5,IE000ZYRH0Q7,IE000ZYRH0Q7,2026-04-26,89.46,89.46,"1,000.00",11.18
6,IE00B88WFS66,IE00B88WFS66,2025-09-23,265.93,265.93,"2,000.00",7.52
7,IE00B88WFS66,IE00B88WFS66,2025-10-04,130.19,130.19,"1,000.00",7.68
8,IE00B88WFS66,IE00B88WFS66,2026-01-14,122.95,122.95,"1,000.00",8.13
9,IE00B88WFS66,IE00B88WFS66,2026-02-24,56.13,56.13,500.00,8.91


: 

: 

## 5. Movimientos + Resumen de Órdenes

In [ ]:
# Resumen de inversiones por mes y año
client.plot_orders_summary(mode="monthly").show()
client.plot_orders_summary(mode="yearly").show()

orders = client.orders_summary()
yearly = orders["yearly"]
if yearly:
    df_yearly = pd.DataFrame([
        {"Año": k, "Total Invertido (€)": v} for k, v in sorted(yearly.items())
    ])
    print("\n📅 Inversiones por Año:")
    display(df_yearly.style.format({"Total Invertido (€)": "€{:,.2f}"}))
    print(f"\nTotal histórico: €{sum(yearly.values()):,.2f}")



📅 Inversiones por Año:


,Año,Total Invertido (€)
0,2024,"€6,000.00"
1,2025,"€82,599.75"
2,2026,"€38,928.53"



Total histórico: €127,528.28


: 

: 

## 6. Asset Allocation

In [ ]:
client.plot_asset_allocation().show()


: 

: 

: 

In [ ]:
client.plot_fund_weights().show()


: 

: 

: 

## 7. Evolución Real del Patrimonio

Reconstrucción diaria del valor de la cartera basada en las **fechas reales de ejecución** de cada orden.
Incluye fondos ya vendidos. Muestra patrimonio vs capital invertido a lo largo del tiempo.

In [5]:
client.plot_real_evolution(years=20).show()


In [7]:
# Tabla mensual de snapshots
evolution = client.real_evolution(years=20)


In [13]:
evolution_perfund = client.real_evolution_per_fund()

In [19]:
evolution_perfund['funds'].keys()

dict_keys(['nan', 'Physical Gold USD (Acc)', 'XS2940466316'])

### Verificación: Evolución por Fondo vs NAV × Participaciones

Comprobamos que la rentabilidad calculada por `real_evolution_per_fund` cuadra con los datos NAV del fondo.
Para cada fondo con posición activa, el valor final de la evolución debe coincidir con `participaciones × NAV actual`.

In [ ]:
# Verificación: valor final por fondo vs posiciones × NAV actual
positions = client.positions(live=True)
funds_evo = evolution_perfund.get("funds", {})
invested_evo = evolution_perfund.get("invested_per_fund", {})

TOLERANCE_PCT = 5.0
results = []
for _, row in positions.iterrows():
    isin = row["ISIN"]
    fondo = row.get("Fondo", isin)
    live_val = float(row.get("Valor_Actual", 0) or 0)
    capital = float(row.get("Capital_Invertido", 0) or 0)

    # Buscar en evolución por fondo (nombre o ISIN)
    fund_series = funds_evo.get(fondo) or funds_evo.get(isin)
    inv_series = invested_evo.get(fondo) or invested_evo.get(isin)

    evo_val = fund_series[-1]["value"] if fund_series else 0.0
    evo_inv = inv_series[-1]["invested"] if inv_series else 0.0
    diff_pct = abs(evo_val - live_val) / live_val * 100 if live_val > 0 else 0.0
    ok = "✅" if diff_pct <= TOLERANCE_PCT else "❌"

    results.append({
        "Fondo": fondo[:35],
        "ISIN": isin,
        "Pos. Live (€)": round(live_val, 2),
        "Evo. Final (€)": round(evo_val, 2),
        "Evo. Inv (€)": round(evo_inv, 2),
        "Diff %": round(diff_pct, 2),
        "OK": ok,
    })

df_check = pd.DataFrame(results)
total_live = df_check["Pos. Live (€)"].sum()
total_evo = df_check["Evo. Final (€)"].sum()
total_diff = abs(total_evo - total_live) / total_live * 100 if total_live > 0 else 0.0

print(f"📊 Verificación NAV: {sum(1 for r in results if r['OK']=='✅')}/{len(results)} fondos OK (tolerancia {TOLERANCE_PCT}%)")
print(f"   Total Live: €{total_live:,.2f}  |  Total Evo: €{total_evo:,.2f}  |  Diff: {total_diff:.2f}%")

display(
    df_check.style
    .format({"Pos. Live (€)": "€{:,.2f}", "Evo. Final (€)": "€{:,.2f}", "Evo. Inv (€)": "€{:,.2f}", "Diff %": "{:.2f}%"})
    .apply(lambda x: ["background-color: #1a3a1a" if v == "✅" else "background-color: #3a1a1a" if v == "❌" else "" for v in x], axis=1)
)

In [12]:
pd.DataFrame(evolution['series'])

,date,value,invested
0,2024-10-27,"2,514.36","2,500.00"
1,2024-10-28,"2,514.64","2,500.00"
2,2024-10-29,"2,514.90","2,500.00"
3,2024-10-30,"2,515.19","2,500.00"
4,2024-10-31,"2,515.19","2,500.00"
...,...,...,...
473,2026-05-04,"140,890.25","137,934.03"
474,2026-05-05,"141,769.04","137,934.03"
475,2026-05-06,"143,572.05","137,934.03"
476,2026-05-07,"143,284.57","137,934.03"


In [ ]:

if evolution["monthly"]:
    df_monthly_evo = pd.DataFrame(evolution["monthly"])
    df_monthly_evo = df_monthly_evo.sort_values("date", ascending=False)
    display(
        df_monthly_evo[["label", "value", "invested", "gain", "gain_pct", "mom"]]
        .rename(columns={
            "label": "Mes", "value": "Patrimonio (€)", "invested": "Invertido (€)",
            "gain": "Ganancia (€)", "gain_pct": "Ganancia (%)", "mom": "MoM (%)"
        })
        .head(24)
        .style.format({
            "Patrimonio (€)": "€{:,.2f}", "Invertido (€)": "€{:,.2f}",
            "Ganancia (€)": "€{:,.2f}", "Ganancia (%)": "{:.2f}%",
            "MoM (%)": "{:.2f}%",
        }).background_gradient(subset=["Ganancia (%)"], cmap="RdYlGn", vmin=-10, vmax=10)
    )


## 8. Evolución Real por Fondo

Desglose del valor diario de cada fondo individual (stacked area chart).

In [ ]:
client.plot_per_fund_evolution(years=20).show()


: 

: 

: 

## 9. Benchmark vs MSCI World

Comparación de la exposición sectorial y geográfica de la cartera vs el benchmark.

In [ ]:
client.plot_benchmark_sectors().show()
client.plot_benchmark_regions().show()


: 

: 

: 

In [ ]:
# Fund metrics table
fm = client.fund_metrics()
if not fm.empty:
    display(fm.style.format(precision=2))

: 

: 

: 

## 10. Detalle Individual de Fondo

Selecciona un ISIN de tu cartera para ver sectores, regiones y top holdings.

In [ ]:
# Elige un ISIN de tu cartera
FUND_ISIN = pos["ISIN"].iloc[0] if not pos.empty else "IE00B4L5Y983"
print(f"Detalle del fondo: {FUND_ISIN}")

detail = client.fund_details(FUND_ISIN)
info_rows = detail[~detail["Metric"].str.startswith(("sector_", "country_", "holding_"))]
holding_rows = detail[detail["Metric"].str.startswith("holding_")].copy()

print("\n📋 Información general:")
display(info_rows)

client.plot_fund_sectors(FUND_ISIN).show()
client.plot_fund_regions(FUND_ISIN).show()

if not holding_rows.empty:
    holding_rows["Holding"] = holding_rows["Metric"].str.replace("holding_", "")
    holding_rows["Peso"] = pd.to_numeric(holding_rows["Value"], errors="coerce")
    print("\n🏢 Top Holdings:")
    display(holding_rows[["Holding", "Peso"]].reset_index(drop=True))


: 

: 

: 

## 11. Evolución Temporal (NAV base 100)

Crecimiento porcentual acumulado normalizado a base 100.

In [ ]:
client.plot_history_base100(years=5).show()


: 

: 

: 

In [ ]:
client.plot_history_nav(years=5).show()


: 

: 

: 

## 12. Métricas de Evolución por Fondo

Rentabilidad Total, CAGR, Volatilidad, Sharpe, Alpha, Beta por fondo.

In [ ]:
evo_metrics = client.evolution_metrics(years=5)

if not evo_metrics.empty:
    print(f"Benchmark: {evo_metrics.attrs.get('benchmark', 'N/A')}")
    print(f"Período: {evo_metrics.attrs.get('years', 5)} años | Risk-free: {evo_metrics.attrs.get('risk_free_annual', 0.03)*100:.1f}%")
    display(
        evo_metrics.style.format({
            "Rentab_Total_Pct": "{:.2f}%",
            "CAGR_Pct": "{:.2f}%",
            "Volatilidad_Pct": "{:.2f}%",
            "Sharpe": "{:.3f}",
            "Alpha_Pct": "{:.2f}%",
            "Beta": "{:.3f}",
            "Peso_Cartera_Pct": "{:.2f}%",
        }).background_gradient(subset=["Sharpe"], cmap="RdYlGn", vmin=-0.5, vmax=1.5)
    )

client.plot_evolution_metrics(years=5, metric="CAGR_Pct").show()
client.plot_evolution_metrics(years=5, metric="Sharpe").show()


: 

: 

: 

## 13. Retornos Anuales (Heatmap)

Rentabilidad de cada fondo por año natural (enero → diciembre).

In [ ]:
client.plot_annual_returns(years=10).show()


: 

: 

: 

## 14. Correlaciones

Matriz de correlación de Pearson entre fondos (5 años de datos diarios).

In [ ]:
client.plot_correlation(years=5).show()

# Pares más y menos correlacionados
corr = client.correlation(years=5)
if not corr.empty:
    pairs = []
    for i in range(len(corr.columns)):
        for j in range(i + 1, len(corr.columns)):
            val = corr.iloc[i, j]
            if not np.isnan(val):
                pairs.append({
                    "Par": f"{corr.columns[i][:20]} — {corr.columns[j][:20]}",
                    "Correlación": corr.iloc[i, j],
                })
    df_pairs = pd.DataFrame(pairs).sort_values("Correlación")
    print("\n🔻 Menos correlacionados:")
    display(df_pairs.head(5))
    print("\n🔺 Más correlacionados:")
    display(df_pairs.tail(5))


: 

: 

: 

## 15. Simulador: Añadir Fondo

Simula el impacto de añadir un nuevo fondo con un importe dado.

In [ ]:
# Configura la simulación
SIM_ISIN = "IE00B4L5Y983"  # iShares MSCI World
SIM_AMOUNT = 10_000  # €

sim = client.simulate_addition(SIM_ISIN, SIM_AMOUNT)
meta = sim.get("metadata", {})
print(f"Simulando añadir €{SIM_AMOUNT:,.0f} de {SIM_ISIN}")
if meta:
    print(f"Fondo añadido:     {meta.get('added_name', SIM_ISIN)}")
    print(f"Total actual:      €{meta.get('current_total', 0):,.2f}")
    print(f"Total simulado:    €{meta.get('simulated_total', 0):,.2f}")
    print()

if "metrics" in sim:
    display(sim["metrics"])

client.plot_simulation_weights(SIM_ISIN, SIM_AMOUNT).show()


: 

: 

: 

## 16. Simulador: Rebalanceo

Simula un rebalanceo de la cartera con pesos objetivo.

In [ ]:
# Define target weights (example: equal weight)
n_funds = len(pos)
equal_weight = round(100 / n_funds, 2) if n_funds > 0 else 0
target_weights = {row["ISIN"]: equal_weight for _, row in pos.iterrows()}

print(f"Simulando rebalanceo equiponderado ({equal_weight:.1f}% cada fondo)")
print(f"{'='*50}")

try:
    rebal = client.simulate_rebalance(target_weights)

    if "funds" in rebal:
        df_rebal = pd.DataFrame(rebal["funds"])
        if "weight_before" in df_rebal.columns and "weight_after" in df_rebal.columns:
            fig = go.Figure()
            fig.add_trace(go.Bar(name="Antes", x=df_rebal["name"], y=df_rebal["weight_before"], marker_color="#4fc3f7"))
            fig.add_trace(go.Bar(name="Después", x=df_rebal["name"], y=df_rebal["weight_after"], marker_color="#66bb6a"))
            fig.update_layout(barmode="group", title="Rebalanceo: Pesos Antes vs Después",
                              template="plotly_dark", height=400, xaxis_tickangle=-45)
            fig.show()

    # Historical comparison
    if "history_current" in rebal and "history_simulated" in rebal:
        h_curr = rebal["history_current"]
        h_sim = rebal["history_simulated"]
        if h_curr and h_sim:
            df_hc = pd.DataFrame(h_curr)
            df_hs = pd.DataFrame(h_sim)
            if "date" in df_hc.columns and "value" in df_hc.columns:
                fig = go.Figure()
                fig.add_trace(go.Scatter(x=df_hc["date"], y=df_hc["value"], name="Actual", line=dict(color="#4fc3f7")))
                fig.add_trace(go.Scatter(x=df_hs["date"], y=df_hs["value"], name="Rebalanceada", line=dict(color="#66bb6a")))
                fig.update_layout(title="Evolución Histórica: Actual vs Rebalanceada (Base 100)",
                                  template="plotly_dark", height=400, hovermode="x unified")
                fig.show()
except Exception as e:
    print(f"Error en simulación de rebalanceo: {e}")

: 

: 

: 

## 17. Proyección What-If

Proyecta el valor futuro de la cartera usando CAGR ± σ con aportaciones anuales.

In [ ]:
# Parámetros de proyección
HORIZON_YEARS = 10
ANNUAL_CONTRIBUTION = 12_000  # € anuales adicionales
LOOKBACK_YEARS = 5
SIGMA_LEVEL = 1.0

client.plot_projection(
    years=LOOKBACK_YEARS,
    horizon=HORIZON_YEARS,
    annual_contribution=ANNUAL_CONTRIBUTION,
    sigma_level=SIGMA_LEVEL,
).show()


: 

: 

: 

## 18. Tax Optimizer (FIFO)

Optimización fiscal para retiradas: selecciona los lotes con menor plusvalía.

In [ ]:
TARGET_WITHDRAWAL = 50_000  # € que quiero retirar

tax_plan = client.tax_optimize(TARGET_WITHDRAWAL)

if not tax_plan.empty:
    print(f"Plan de retirada óptimo para €{TARGET_WITHDRAWAL:,.0f}")
    print(f"  Importe retirado:     €{tax_plan.attrs.get('withdrawn_amount', 0):,.2f}")
    print(f"  Ganancia patrimonial: €{tax_plan.attrs.get('total_capital_gain', 0):,.2f}")
    print(f"  Impuestos estimados:  €{tax_plan.attrs.get('estimated_tax', 0):,.2f}")
    print(f"  Neto tras impuestos:  €{tax_plan.attrs.get('net_amount', 0):,.2f}")
    display(tax_plan)

client.plot_tax_brackets(TARGET_WITHDRAWAL).show()


: 

: 

: 

## 19. Análisis de Traspasos

Fondos traspasables para diferir impuestos (Art. 94 LIRPF) + optimizador combinado.

In [ ]:
# Análisis de traspasos por fondo
try:
    traspaso = client.traspaso_analysis()
    if traspaso:
        df_t = pd.DataFrame(traspaso)
        print("📋 Análisis de Fondos Traspasables (Art. 94 LIRPF)")
        print("=" * 60)
        display(df_t)

        # Summary
        if "Plusvalia_Latente" in df_t.columns:
            total_latent = df_t["Plusvalia_Latente"].sum()
            print(f"\nPlusvalía latente total diferible: €{total_latent:,.2f}")
        if "Ahorro_Traspaso" in df_t.columns:
            total_saving = df_t["Ahorro_Traspaso"].sum()
            print(f"Ahorro fiscal potencial (vs vender): €{total_saving:,.2f}")
    else:
        print("No hay datos de análisis de traspasos.")
except Exception as e:
    print(f"Error: {e}")

: 

: 

: 

In [ ]:
# Optimizador combinado: Traspaso + Reembolso
TRASPASO_AMOUNT = 30_000  # € que quiero retirar de forma óptima

try:
    opt_result = client.optimize_withdrawal_via_traspaso(TRASPASO_AMOUNT)

    if opt_result:
        print(f"🎯 Optimización de retirada de €{TRASPASO_AMOUNT:,.0f}")
        print("=" * 60)

        # Scenario comparison
        if "scenario_direct" in opt_result and "scenario_optimized" in opt_result:
            direct = opt_result["scenario_direct"]
            optimized = opt_result["scenario_optimized"]
            print(f"\n{'Escenario':.<30} {'Directo':>12} {'Optimizado':>12} {'Ahorro':>12}")
            print("-" * 70)
            for key in ["impuesto_total", "neto_retirado", "plusvalia_diferida"]:
                d_val = direct.get(key, 0)
                o_val = optimized.get(key, 0)
                saving = d_val - o_val if "impuesto" in key else o_val - d_val
                print(f"  {key:.<28} €{d_val:>10,.2f} €{o_val:>10,.2f} €{saving:>10,.2f}")

        # Steps
        if "steps" in opt_result:
            print(f"\n📝 Plan paso a paso ({len(opt_result['steps'])} operaciones):")
            for i, step in enumerate(opt_result["steps"][:10], 1):
                tipo = step.get("tipo", "")
                fondo = step.get("Fondo", "")[:30]
                importe = step.get("Importe", 0)
                print(f"  {i}. [{tipo}] {fondo} → €{importe:,.2f}")
except Exception as e:
    print(f"Error en optimización de traspasos: {e}")

: 

: 

: 

## 20. Performance + Diagnósticos

In [ ]:
# Performance metrics
perf = client.performance(years=5)
if not perf.empty:
    print("📊 Performance del Portfolio (5Y)")
    print("=" * 40)
    for _, row in perf.iterrows():
        print(f"  {row['Metric']:.<30} {row['Value']}")
else:
    print("No hay métricas de performance disponibles.")

: 

: 

: 

In [ ]:
# Diagnósticos de cobertura de datos
diag = client.diagnostics(years=5)
if not diag.empty:
    print("🔍 Diagnóstico de Datos")
    print("=" * 40)
    display(diag)
else:
    print("No hay datos de diagnóstico.")

: 

: 

: 